<a href="https://colab.research.google.com/github/ajaykumar080286/GENAI_LangChain/blob/main/17_langchain_retrievers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
#!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia

**Wikipedia Retriever**

In [17]:
from langchain_community.retrievers import WikipediaRetriever

In [18]:
retriever=WikipediaRetriever(top_k_results=2,lang="en")

In [19]:
query="is TCS is IT company"


In [20]:
docs= retriever.invoke(query)

In [21]:
for i,doc in enumerate(docs):
  print("-----Result-----",i+1)
  print("\n")
  print(doc)

-----Result----- 1


page_content='Tranzum Courier Service (TCS) is a Pakistani courier and logistics company based in Karachi. It was founded as a joint venture with DHL by a former Pakistan International Airlines (PIA) flight engineer, Khalid Nawaz Awan and his brother Sadiq Awan in 1983.


== History ==
TCS was founded in 1983 by Khalid Awan and his elder brother Sadiq Awan, as a joint venture with DHL. Before joining TCS, Khalid Awan worked as a flight engineer with Pakistan International Airlines.
In its early years, TCS faced regulatory challenges under the 1898 Post Office Act, which limited the delivery of "letters" to the national post office. TCS contended that its services included "business-related documents" and international packages, leading to a joint venture arrangement where DHL owned 51% of the international delivery operations, and TCS retained full ownership of local operations.
In 1985, TCS secured a contract with the Pakistan Banking Council to provide overnight 

**Vector Store Retriever**

In [22]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

In [23]:
import os
from getpass import getpass

# Securely set your OpenAI API key if not already set
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")


Enter your OpenAI API Key: ··········


In [24]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [28]:
import os
from getpass import getpass

# Securely set your OpenAI API key if not already set
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

Enter your OpenAI API Key: ··········


In [29]:
# Step 2: Initialize embedding model
embedding_model = OpenAIEmbeddings()

# Step 3: Create Chroma vector store in memory
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

In [30]:
# Step 4: Convert vectorstore into a retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [31]:
query = "What is Chroma used for?"
result=retriever.invoke(query)

In [32]:
for i,doc in enumerate(result):
  print("-----result----", i+1)
  print(doc)

-----result---- 1
page_content='Chroma is a vector database optimized for LLM-based search.'
-----result---- 2
page_content='LangChain helps developers build LLM applications easily.'


In [33]:
results = vector_store.similarity_search(query, k=2)

In [34]:
for i,doc in enumerate(result):
  print("-----result----", i+1)
  print(doc)

-----result---- 1
page_content='Chroma is a vector database optimized for LLM-based search.'
-----result---- 2
page_content='LangChain helps developers build LLM applications easily.'


***MMR***

In [35]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [36]:
docs

[Document(metadata={}, page_content='LangChain makes it easy to work with LLMs.'),
 Document(metadata={}, page_content='LangChain is used to build LLM based applications.'),
 Document(metadata={}, page_content='Chroma is used to store and search document embeddings.'),
 Document(metadata={}, page_content='Embeddings are vector representations of text.'),
 Document(metadata={}, page_content='MMR helps you get diverse results when doing similarity search.'),
 Document(metadata={}, page_content='LangChain supports Chroma, FAISS, Pinecone, and more.')]

In [37]:
from langchain_community.vectorstores import FAISS

In [38]:
# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings()
)

In [39]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 1}  # k = top results, lambda_mult = relevance-diversity balance
)

In [40]:
query = "What is langchain?"
results = retriever.invoke(query)

In [41]:
for i ,doc in enumerate(results):
  print("---------Result-------", i+1)
  print(doc)

---------Result------- 1
page_content='LangChain is used to build LLM based applications.'
---------Result------- 2
page_content='LangChain makes it easy to work with LLMs.'
---------Result------- 3
page_content='LangChain supports Chroma, FAISS, Pinecone, and more.'


***Multiquery Retriever***

In [42]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI


In [43]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [44]:
all_docs

[Document(metadata={'source': 'H1'}, page_content='Regular walking boosts heart health and can reduce symptoms of depression.'),
 Document(metadata={'source': 'H2'}, page_content='Consuming leafy greens and fruits helps detox the body and improve longevity.'),
 Document(metadata={'source': 'H3'}, page_content='Deep sleep is crucial for cellular repair and emotional regulation.'),
 Document(metadata={'source': 'H4'}, page_content='Mindfulness and controlled breathing lower cortisol and improve mental clarity.'),
 Document(metadata={'source': 'H5'}, page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.'),
 Document(metadata={'source': 'I1'}, page_content='The solar energy system in modern homes helps balance electricity demand.'),
 Document(metadata={'source': 'I2'}, page_content='Python balances readability with power, making it a popular system design language.'),
 Document(metadata={'source': 'I3'}, page_content='Photosynthesis enables plants

In [45]:
# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [46]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [47]:
query = "How to improve energy levels and maintain balance?"
results = similarity_retriever.invoke(query)

In [48]:
for i ,doc in enumerate(results):
  print("------result----", i+1)
  print(doc)

------result---- 1
page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.' metadata={'source': 'H5'}
------result---- 2
page_content='Mindfulness and controlled breathing lower cortisol and improve mental clarity.' metadata={'source': 'H4'}
------result---- 3
page_content='Regular walking boosts heart health and can reduce symptoms of depression.' metadata={'source': 'H1'}
------result---- 4
page_content='Deep sleep is crucial for cellular repair and emotional regulation.' metadata={'source': 'H3'}
------result---- 5
page_content='The solar energy system in modern homes helps balance electricity demand.' metadata={'source': 'I1'}


In [49]:
from langchain_openai import ChatOpenAI
import langchain_community
from langchain_classic.retrievers.multi_query import MultiQueryRetriever


In [53]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=ChatOpenAI(model="gpt-4o")
)

In [54]:
# Query
query = "How to improve energy levels and maintain balance?"

In [55]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [56]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 4 ---
Deep sleep is crucial for cellular repair and emotional regulation.

--- Result 5 ---
The solar energy system in modern homes helps balance electricity demand.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Deep sleep is crucial for cellular repair and emotional regulation.

--- Result 4 ---
Regular walking boosts heart health and can reduce symptoms of depression.



**ContextualCompressionRetriever**

In [58]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [59]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [60]:
# Create a FAISS vector store from the documents
embedding_model = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embedding_model)

In [61]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [62]:
# Set up the compressor using an LLM
llm = ChatOpenAI(model="gpt-4o")
compressor = LLMChainExtractor.from_llm(llm)

In [63]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [64]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [65]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)



--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.

--- Result 3 ---
Photosynthesis does not occur in animal cells.
